## word2vec 속도 개선

### word2vec 개선 1
* CBOW 모델은 작은 말뭉치를 다룰 때는 문제될 것이 없음
* 거대한 말뭉치를 다루게 될 경우 문제가 발생
    * 어휘가 100만 개, 은릭층의 뉴런이 100개인 모델을 가정 시 두 계산이 병목
        * 입력층의 원핫 표현과 가중치 행렬 $W_{in}$의 곱 계산  
        => Embedding 계층을 도입하는 것으로 해결
        * 은닉층과 가중치 행렬 $W_{out}$의 곱 및 Softmax 계층의 계산  
        => 네거티브 샘플링이라는 손실 함수로 해결

#### Embedding 계층
* 가중치 매개변수로부터 단어 ID에 해당하는 행을 추출하는 계층
* Embedding 계층에 단어 임베딩을 저장하는 것

#### Embedding 계층 구현
* 행렬에서 특정 행을 추출하려면 원하는 행을 명시하면 끝

In [8]:
import numpy as np
W = np.arange(21).reshape(7, 3)

print(W)
print(W[2])
print(W[5])

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]
 [12 13 14]
 [15 16 17]
 [18 19 20]]
[6 7 8]
[15 16 17]


In [9]:
# 가중치 W로부터 여러 행을 한꺼번에 추출
idx = np.array([1, 0, 3, 0])

print(W[idx])

[[ 3  4  5]
 [ 0  1  2]
 [ 9 10 11]
 [ 0  1  2]]


* 인덱스 4개를 한 번에 추출
* 미니배치 처리를 가정했을 경우의 구현

In [10]:
# Embedding 계층
class Embedding:
    def __init__(self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.idx = None

    def forward(self, idx):
        W, = self.params
        self.idx = idx
        out = W[idx]
        return out


* 역전파는 앞 층으로부터 전해진 기울기를 다음 층으로 그대로 흘려줌
* 다만, 앞 층으로부터 전해진 기울기를 가중치 기울기 dW의 특정행(idx번째 행)에 설정

In [11]:
def backward(self, dout):
    dW, = self.grads
    dW[...] = 0
    dW[self.idx] = dout # 나쁜 예
    return None

* 가중치 기울기 dW를 꺼낸 다음, dW[...]의 원소를 0으로 덮어씀 (dW의 형상은 유지)
* 앞 층에서 전해진 기울기 dout을 idx번째 행에 할당
* 사실 갱신하려는 행 번호와 기울기를 따로 저장해두면 이 정보로부터 가중치의 특정 행만 갱신할 수 있음

* 위의 코드는 idx의 원소가 중복될 때 문제가 발생  
-> 중복 문제를 해결하려면 할당이 아닌 더하기를 해야 함  
=> 즉, dh의 각 행의 값을 dW에 더해줌 (더하는 이유는  gradient를 누적해야 함)

In [12]:
def backward(self, dout):
    dW, = self.grads
    dW[...] = 0

    for i, word_id in enumerate(self.idx):
        dW[word_id] += dout[i]
    # 혹은
    # np.add.at(dW, self.idx, dout)

    return None

* np.add.at(A, idx, B)는 B를 A의 idx번째 행에 더해줌
* 일반적으로 for문보다 numpy의 내장 메서드를 사용하는 편이 더 빠름

### word2vec 개선 2
* 은닉층 이후의 처리에서의 병목을 해결해야 함  
=> 네거티브 샘플링 기법 사용
* Softmax 대신 네거티브 샘플링을 이용하면 어휘가 아무리 많아져도 계산량을 낮은 수준에서 일정하게 억제할 수 있음

#### 은닉층 이후의 계산의 문제점
* 어휘가 100만 개, 은닉층 뉴런이 100개일 때의 word2vec(CBOW 모델)
    * 거대한 행렬을 곱하는 문제
    * softmax 계산량의 증가

#### 다중 분류에서 이진 분류
* 네거티브 샘플링 기법의 핵심은 이진 분류
    * 정확하게는 다중 분류에서 이진 분류로 근사
* 생각해야 할 것은 다중 분류 문제를 이진 분류 방식으로 해결하는 것  
=> 이러면 출력층에는 뉴런을 하나만 준비하면 됨  
* 은닉층과 출력 측의 가중치 행렬의 내적은 해당하는 열만을 추출하고 추출된 벡터와 은닉층 뉴런과의 내적을 계산하면 됨

#### 시그모이드 함수와 교차 엔트로피 오차
* 이진 분류 문제를 신경망으로 풀려면 
    * 점수에 시그모이드 함수를 적용해 확률로 변환
    * 손실을 구할 때는 손실 함수로 교차 엔트로피 오차를 사용
* 시그모이드 계층과 교차 엔트로피 오차 계층의 역전파의 y - t에 주목
    * 오차가 크면 크게 학습, 작으면 작게 학습

#### 다중 분류에서 이진 분류로 (구현)
* 후반부를 더 단순하게 만들기 위해 Embedding Dot 계층을 도입
    * Embedding 계층 + dot 연산
* 은닉층 뉴런 h는 Embedding Dot 계층을 거쳐 Sigmoid with Loss 계층을 통과  
=> 간단해짐

In [13]:
class EmbeddingDot:
    def __init__(self, W):
        self.embed = Embedding(W)
        self.params = self.embed.params
        self.grads = self.embed.grads
        self.cache = None

    def forward(self, h, idx):
        target_W = self.embed.forward(idx)
        out = np.sum(target_W * h, axis=1)

        self.cache = (h, target_W)
        return out

    def backward(self, dout):
        h, target_W = self.cache
        dout = dout.reshape(dout.shape[0], 1)

        dtarget_W = dout * h
        self.embed.backward(dtarget_W)
        dh = dout * target_W
        return dh

* params에 매개변수 저장
* grads에 기울기를 저장
* embed는 Embedding 계층
* cache는 순전파 시 계산 결과를 임시 저장

* forward 메서드는 인수로 뉴런(h)과 단어 ID의 numpy 배열(idx)를 받음
    * 미니배치 처리를 가정했기 때문에 idx는 배열
* forward 메서드에서는 Embedding 계층의 forward(idx)를 호출한 다음 내적을 계산
    * 내적 계산은 np.sum(self.target_W * h, axis=1)
  
=> Embedding Dot 계층의 순전파

#### 네거티브 샘플링
* 원하는 것은 긍정적인 예에 대해서는 Sigmoid 계층의 출력을 1에 가깝게 만들고, 부정적인 예에 대해서는 Sigmoid 계층의 출력을 0엑 가깝게 만드는 것
* 모든 부정적인 예를 대상으로 이진 분류를 학습시키는 것은 어려움  
-> 적은 수의 부정적 예를 샘플링해 사용  
=> 네거티브 샘플링

* 네거티브 샘플링 기법은 긍정적 예를 타깃으로 한 경우의 손실을 구함
* 부정적 예를 몇 개 샘플링해 부정적 예에 대해서도 손실을 구함
* 각 각의 데이터의 손실을 더한 값을 최종 손실로 함

#### 네거티브 샘플링의 샘플링 기법
* 무작위로 샘플링하는 것보다 말뭉치의 통계 데이터를 기초로 샘플링하는 것이 더 좋음  
=> 말뭉치에서 자주 등장하는 단어를 많이 추출하고 드물게 등장하는 단어를 적게 추출하는 것
* 말뭉치에서 단어 빈도를 기준으로 샘플링하려면 각 단어의 출현 횟수를 구해 확률분포로 나타내고 이를 토대로 샘플링

* 네거티브 샘플링에서 부정적 예를 가능한 많이 다루는 것이 좋음
* 그러나 계산량 문제로 인해 적은 수로 한정해야 함
* 흔한 단어를 잘 처리하는 편이 좋은 겨려과로 이어짐

In [14]:
import numpy as np

# 0~9 중 하나를 무작위로 샘플링
print(np.random.choice(10))
print(np.random.choice(10))

# words에서 하나만 무작위로 샘플링
words = ['you', 'say', 'goodbye', 'I', 'hello', '.']
print(np.random.choice(words))

# 5개만 무작위로 샘플링(중복 있음)
print(np.random.choice(words, size=5))

# 5개만 무작위로 샘플링(중복 없음)
print(np.random.choice(words, size=5, replace=False))

# 확률 분포에 따라 샘플링
p = [0.5, 0.1, 0.05, 0.2, 0.05, 0.1]
print(np.random.choice(words, p=p))

9
4
say
['hello' 'I' 'say' 'hello' 'goodbye']
['you' '.' 'goodbye' 'say' 'hello']
you


* word2vec의 네거티브 샘플링에서는 앞의 확률분포에서 0.75를 제곱하는 것을 권고
$$
P'(w_i) = \frac{P(w_i)^{0.75}}{\sum_j^n P(w_j)^{0.75}}
$$
* 다만, 수정 후에도 확률의 총합은 1이 되어야 하므로 분모로는 수정 후 확률분포의 총합이 필요  
=> 출현 확률이 낮은 단어를 버리지 않기 위해서, 원래 확률이 낮은 단어의 확률을 살짝 높일 수 있음

In [15]:
import collections

class UnigramSampler:
    def __init__(self, corpus, power, sample_size):
        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        self.word_p = np.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        self.word_p = np.power(self.word_p, power)
        self.word_p /= np.sum(self.word_p)

    def get_negative_sample(self, target):
        batch_size = target.shape[0]

        negative_sample = np.zeros((batch_size, self.sample_size), dtype=np.int32)

        for i in range(batch_size):
            p = self.word_p.copy()
            target_idx = target[i]
            p[target_idx] = 0
            p /= p.sum()
            negative_sample[i, :] = np.random.choice(self.vocab_size, size=self.sample_size, replace=False, p=p)

        return negative_sample

In [16]:
corpus = np.array([0, 1, 2, 3, 4, 1, 2, 3])
power = 0.75
sample_size = 2

sampler = UnigramSampler(corpus, power, sample_size)
target = np.array([1, 3, 0])
negative_sample = sampler.get_negative_sample(target)
print(negative_sample)

[[2 0]
 [1 0]
 [4 2]]


* 긍정적 예로 [1, 3, 0]이라는 3개의 데이터
* 각각의 데이터에 대해서 부정적 예를 2개씩 샘플링

#### 네거티브 샘플링 구현

In [17]:
import os, sys
sys.path.append('..')
from common.layers import SigmoidWithLoss

class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)]
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)]

        self.params, self.grads = [], []
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, h, target):
        batch_size = target.shape[0]
        negative_sample = self.sampler.get_negative_sample(target)

        # 긍정적 예 순전파
        score = self.embed_dot_layers[0].forward(h, target)
        correct_label = np.ones(batch_size, dtype=np.int32)
        loss = self.loss_layers[0].forward(score, correct_label)

        # 부정적 예 순전파
        negative_label = np.zeros(batch_size, dtype=np.int32)
        for i in range(self.sample_size):
            negative_target = negative_sample[:, i]
            score = self.embed_dot_layers[1 + i].forward(h, negative_target)
            loss += self.loss_layers[1 + i].forward(score, negative_label)

        return loss

    def backward(self, dout=1):
        dh = 0
        for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
            dscore = l0.backward(dout)
            dh += l1.backward(dscore)

        return dh

### 개선판 word2vec 학습
* PTB 데이터셋을 사용해 학습하고 더 실용적인 단어의 분산 표현 얻기

#### CBOW 모델 구현
* 개선된 CBOW 클래스

In [18]:
import sys
sys.path.append('..')
from common.np import *  # import numpy as np
from common.layers import Embedding

class CBOW:
    def __init__(self, vocab_size, hidden_size, window_size, corpus):
        V, H = vocab_size, hidden_size

        # 가중치 초기화
        W_in = 0.01 * np.random.randn(V, H).astype('f')
        W_out = 0.01 * np.random.randn(V, H).astype('f')

        # 계층 생성
        self.in_layers = []
        for i in range(2 * window_size):
            layer = Embedding(W_in)  # Embedding 계층 사용
            self.in_layers.append(layer)
        self.ns_loss = NegativeSamplingLoss(W_out, corpus, power=0.75, sample_size=5)

        # 모든 가중치와 기울기를 배열에 모은다.
        layers = self.in_layers + [self.ns_loss]
        self.params, self.grads = [], []
        for layer in layers:
            self.params += layer.params
            self.grads += layer.grads

        # 인스턴스 변수에 단어의 분산 표현을 저장한다.
        self.word_vecs = W_in

    def forward(self, contexts, target):
        h = 0
        for i, layer in enumerate(self.in_layers):
            h += layer.forward(contexts[:, i])
        h *= 1 / len(self.in_layers)
        loss = self.ns_loss.forward(h, target)
        return loss

    def backward(self, dout=1):
        dout = self.ns_loss.backward(dout)
        dout *= 1 / len(self.in_layers)
        for layer in self.in_layers:
            layer.backward(dout)
        return None


#### CBOW 모델 학습

In [ ]:
import sys
sys.path.append('..')
import numpy as np

import pickle
from common.trainer import Trainer
from common.optimizer import Adam
from common.util import create_contexts_target
from dataset import ptb


# 하이퍼파라미터 설정
window_size = 5
hidden_size = 100
batch_size = 100
max_epoch = 10

# 데이터 읽기
corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)

contexts, target = create_contexts_target(corpus, window_size)

# 모델 등 생성
model = CBOW(vocab_size, hidden_size, window_size, corpus)
# model = SkipGram(vocab_size, hidden_size, window_size, corpus)
optimizer = Adam()
trainer = Trainer(model, optimizer)

# 학습 시작
trainer.fit(contexts, target, max_epoch, batch_size)
trainer.plot()

# 나중에 사용할 수 있도록 필요한 데이터 저장
word_vecs = model.word_vecs
params = {}
params['word_vecs'] = word_vecs.astype(np.float16)
params['word_to_id'] = word_to_id
params['id_to_word'] = id_to_word
pkl_file = 'cbow_params.pkl'  # or 'skipgram_params.pkl'
with open(pkl_file, 'wb') as f:
    pickle.dump(params, f, -1)

#### CBOW 모델 평가

In [20]:
import sys
sys.path.append('..')
from common.util import most_similar, analogy
import pickle


pkl_file = 'cbow_params.pkl'
# pkl_file = 'skipgram_params.pkl'

with open(pkl_file, 'rb') as f:
    params = pickle.load(f)
    word_vecs = params['word_vecs']
    word_to_id = params['word_to_id']
    id_to_word = params['id_to_word']

# 가장 비슷한(most similar) 단어 뽑기
querys = ['you', 'year', 'car', 'toyota']
for query in querys:
    most_similar(query, word_to_id, id_to_word, word_vecs, top=5)

# 유추(analogy) 작업
print('-'*50)
analogy('king', 'man', 'queen',  word_to_id, id_to_word, word_vecs)
analogy('take', 'took', 'go',  word_to_id, id_to_word, word_vecs)
analogy('car', 'cars', 'child',  word_to_id, id_to_word, word_vecs)
analogy('good', 'better', 'bad',  word_to_id, id_to_word, word_vecs)

C:\Users\기현\AppData\Local\Temp\ipykernel_14452\3720099090.py:11: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  params = pickle.load(f)



[query] you
 we: 0.6103515625
 someone: 0.59130859375
 i: 0.55419921875
 something: 0.48974609375
 anyone: 0.47314453125

[query] year
 month: 0.71875
 week: 0.65234375
 spring: 0.62744140625
 summer: 0.6259765625
 decade: 0.603515625

[query] car
 luxury: 0.497314453125
 arabia: 0.47802734375
 auto: 0.47119140625
 disk-drive: 0.450927734375
 travel: 0.4091796875

[query] toyota
 ford: 0.55078125
 instrumentation: 0.509765625
 mazda: 0.49365234375
 bethlehem: 0.47509765625
 nissan: 0.474853515625
--------------------------------------------------

[analogy] king:man = queen:?
 woman: 5.16015625
 veto: 4.9296875
 ounce: 4.69140625
 earthquake: 4.6328125
 successor: 4.609375

[analogy] take:took = go:?
 went: 4.55078125
 points: 4.25
 began: 4.09375
 comes: 3.98046875
 oct.: 3.90625

[analogy] car:cars = child:?
 children: 5.21875
 average: 4.7265625
 yield: 4.20703125
 cattle: 4.1875
 priced: 4.1796875

[analogy] good:better = bad:?
 more: 6.6484375
 less: 6.0625
 rather: 5.21875
 slow

* word2vec으로 얻은 단어의 분산 표현은 비슷한 단어를 가까이 모을 뿐 아니라, 더 복잡한 패턴을 파악함

In [22]:
analogy('king', 'man', 'queen', word_to_id, id_to_word, word_vecs)


[analogy] king:man = queen:?
 woman: 5.16015625
 veto: 4.9296875
 ounce: 4.69140625
 earthquake: 4.6328125
 successor: 4.609375


### word2vec 남은 주제

#### word2vec을 사용한 애플리케이션의 예
* 자연어 처리 분야에서 단어의 분산 표현이 중요한 이유는 전이 학습 때문
    * 전이 학습은 한 분야에서 배운 지식을 다른 분야에도 적용하는 기법
* 자연어 문제를 풀 때 word2vec의 단어 분산 표현을 처음부터 학습하는 일은 거의 없음
* 큰 말뭉치로 학습 후 분산 표현을 각자의 작업에 이용하는 것

* 단어의 분산 표현은 단어를 고정 길이 벡터로 변환해주는 장점도 있음
* 문장도 단어의 분산 표현을 사용해 고정 길이 벡터로 변환할 수 있음
    * 가장 간단한 방법은 문장의 각 단어를 분산 표현으로 변환하고 그 합을 구함  
        => bag-of-words
    * RNN 사용 시에도 문장을 고정 길이 벡터로 변환할 수 있음
* 자연어로 쓰인 질문을 고정 길이 벡터로 변환할 수 있다면, 그 벡터를 다른 머신러닝 시스템의 입력으로 사용할 수 있음

* 메일을 자동으로 분류하는 시스템, 3단계로 분류
    1. 데이터(메일)를 수집
    2. 3단계의 감정을 나타내는 레이블(긍정적, 중립적, 부정적)을 붙임
    3. word2vec으로 메일을 벡터로 변환
    4. 감정 분석을 수행하는 분류 시스템에 벡터화된 메일과 감정 레이블을 입력해 학습을 수행

#### 단어 벡터 평가 방법
* 분산 표현의 우수성을 실제 애플리케이션과는 분리해 평가하는 것이 일반적  
    => 자주 사용되는 펴악 척도가 단어의 유사성이나 유추 문제를 활용한 평가

* 단어의 유사성 평가
    * 사람이 작성한 단어 유사도를 검증 세트를 이용해 평가
    * 사람이 부여한 점수와 word2vec에 의한 코사인 유사도 점수를 비교해 상관성을 보는 것
* 유추 문제를 활용한 평가
    * 유추 문제를 출제하고 정답률로 우수성을 측정

* 다만, 단어의 분산 표현의 우수함이 애플리케이션에 얼마나 기여하는지는 문제 상황에 따라 다름